# 🏥 Task 1: Phân loại Bệnh nhân với Automated Machine Learning

## Mục tiêu
Sử dụng **Azure AutoML** để tự động tìm thuật toán tốt nhất phân loại bệnh nhân vào `Routing_Class`:
- `0` → Fast-Track (bệnh nhẹ)
- `1` → Regular/Resuscitation (bệnh nặng)

## Dataset: ER_Triage_Dataset
| Feature | Mô tả |
|---------|-------|
| Age | Tuổi bệnh nhân |
| Arrival_Mode | Walk-in / Ambulance |
| Heart_Rate | Nhịp tim (bpm) |
| Pain_Scale | Mức độ đau (1-10) |
| Primary_Symptom | Triệu chứng chính |
| **Routing_Class** | **Target: 0=Fast-Track, 1=Regular** |

## Bước 1: Kiểm tra SDK

In [ ]:
# Kiểm tra phiên bản azure-ai-ml
pip show azure-ai-ml

## Bước 2: Kết nối Azure ML Workspace

In [ ]:
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml import MLClient

try:
    credential = DefaultAzureCredential()
    # Kiểm tra credential có hợp lệ không
    credential.get_token("https://management.azure.com/.default")
    print("✅ DefaultAzureCredential thành công")
except Exception as ex:
    print(f"⚠️ DefaultAzureCredential thất bại: {ex}")
    print("🔄 Đang dùng InteractiveBrowserCredential...")
    credential = InteractiveBrowserCredential()

In [ ]:
# Kết nối workspace từ file config.json
ml_client = MLClient.from_config(credential=credential)
print(f"✅ Đã kết nối workspace: {ml_client.workspace_name}")
print(f"   Subscription: {ml_client.subscription_id}")
print(f"   Resource Group: {ml_client.resource_group_name}")

## Bước 3: Khám phá Dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Đọc dataset
df = pd.read_csv("ERTriage-Data/ER_Triage_Dataset.csv")

print("📊 Thông tin Dataset:")
print(f"   Số mẫu: {len(df)}")
print(f"   Số features: {len(df.columns) - 1}")
print(f"   Cột target: Routing_Class")
print()
print(df.head(10))

In [ ]:
# Phân phối target
print("\n🎯 Phân phối Routing_Class:")
target_counts = df['Routing_Class'].value_counts()
for label, count in target_counts.items():
    category = 'Fast-Track' if label == 0 else 'Regular/Resus'
    print(f"   Class {label} ({category}): {count} bệnh nhân ({count/len(df)*100:.1f}%)")

# Biểu đồ phân phối
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Target distribution
axes[0].bar(['Fast-Track (0)', 'Regular (1)'], target_counts.values, color=['#2196F3', '#FF5722'])
axes[0].set_title('Phân phối Routing_Class')
axes[0].set_ylabel('Số bệnh nhân')

# Arrival Mode
df['Arrival_Mode'].value_counts().plot(kind='bar', ax=axes[1], color=['#4CAF50', '#FF9800'])
axes[1].set_title('Phương thức đến viện')
axes[1].tick_params(axis='x', rotation=0)

# Primary Symptom
df['Primary_Symptom'].value_counts().plot(kind='bar', ax=axes[2], color='steelblue')
axes[2].set_title('Triệu chứng chính')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('eda_charts.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Đã lưu biểu đồ EDA")

## Bước 4: Đăng ký Data Asset (MLTable)

> **Lưu ý**: Bước này chỉ cần thực hiện 1 lần. Dataset đã có trong thư mục `ERTriage-Data/`.

In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

# Đăng ký MLTable data asset
my_data = Data(
    path="./ERTriage-Data",
    type=AssetTypes.MLTABLE,
    description="ER Triage Dataset - Phân loại bệnh nhân khẩn cấp",
    name="ER_Triage_Dataset",
    version="1"
)

ml_client.data.create_or_update(my_data)
print("✅ Đã đăng ký MLTable: ER_Triage_Dataset v1")

## Bước 5: Cấu hình AutoML Job

In [ ]:
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml import Input

# Load MLTable data asset đã đăng ký
my_training_data_input = Input(
    type=AssetTypes.MLTABLE,
    path="azureml:ER_Triage_Dataset:1"
)
print("✅ Đã load training data input")

In [ ]:
from azure.ai.ml import automl

# Cấu hình AutoML Classification Job
classification_job = automl.classification(
    compute="aml-cluster",
    experiment_name="ERTriage-AutoML-Classification",
    training_data=my_training_data_input,
    target_column_name="Routing_Class",      # Cột cần dự đoán
    primary_metric="accuracy",               # Metric chính
    n_cross_validations=5,                   # 5-fold cross validation
    enable_model_explainability=True         # Giải thích model
)

# Giới hạn thời gian và số models
classification_job.set_limits(
    timeout_minutes=60,            # Tổng thời gian tối đa: 60 phút
    trial_timeout_minutes=20,      # Mỗi trial tối đa: 20 phút
    max_trials=5,                  # Thử tối đa 5 models
    enable_early_termination=True  # Dừng sớm nếu không cải thiện
)

# Cài đặt training (bỏ qua LogisticRegression vì đã dùng ở Task 2)
classification_job.set_training(
    blocked_training_algorithms=["LogisticRegression"],  # Không dùng LR
    enable_onnx_compatible_models=True                   # Hỗ trợ ONNX export
)

print("✅ Đã cấu hình AutoML job")
print(f"   Target: Routing_Class")
print(f"   Metric: accuracy")
print(f"   Max trials: 5")
print(f"   Timeout: 60 phút")

## Bước 6: Chạy AutoML Job

In [ ]:
# Submit AutoML job
returned_job = ml_client.jobs.create_or_update(classification_job)

# In link để theo dõi
aml_url = returned_job.studio_url
print(f"🚀 Job đã được submit!")
print(f"   Theo dõi tại: {aml_url}")
print(f"\n⏳ Đang chạy... (có thể mất 20-60 phút)")

In [ ]:
# Chờ job hoàn thành (optional - có thể theo dõi trên Studio)
from azure.ai.ml.entities import Job

ml_client.jobs.stream(returned_job.name)
print("✅ AutoML job hoàn thành!")

## Bước 7: Xem kết quả và Model tốt nhất

In [ ]:
# Lấy model tốt nhất từ AutoML
# Chú ý: Chạy sau khi job hoàn thành

from azure.ai.ml.entities import Model

# Lấy thông tin best run
best_run = ml_client.jobs.get(returned_job.name)
print(f"📊 Kết quả AutoML:")
print(f"   Job Name: {best_run.name}")
print(f"   Status: {best_run.status}")
print(f"   Studio URL: {best_run.studio_url}")
print()
print("💡 Tip: Vào Azure ML Studio → Jobs → ERTriage-AutoML-Classification")
print("        Xem tab 'Models' để thấy model ranking và best model")

## 📋 Kết luận Task 1

AutoML đã tự động:
1. **Thử nhiều thuật toán** (Random Forest, XGBoost, LightGBM, v.v.)
2. **Tiền xử lý dữ liệu** tự động (scaling, encoding)
3. **Chọn model tốt nhất** dựa trên accuracy
4. **Cung cấp model explainability** (feature importance)

> **Ghi chú y tế quan trọng**: Với bài toán triage, **Recall** (sensitivity) quan trọng hơn Accuracy.
> False Negative (bỏ sót bệnh nặng) nguy hiểm hơn False Positive.